# 110 Python intermediate - meta classes
_Kamil Bartocha_

_wersja_ 0.0.1

## Metaklasy w Pythonie

### 1. Co to jest metaklasa?
- Metaklasy to "klasy dla klas".
- Odpowiadają za tworzenie klas, podobnie jak klasy odpowiadają za tworzenie obiektów.
- Domyślną metaklasą w Pythonie jest `type`.



In [13]:
class MyClass:
    pass

print(type(MyClass))  # <class 'type'>


<class 'type'>


In [ ]:
list1 = [1, 2, 3]
for el in list1:
    if el == 2:
        x = list1.pop()
    print(el)

1
2


Tutaj:

`MyClass` to klasa.
`type` to metaklasa odpowiedzialna za stworzenie klasy MyClass.

### 2. Jak działają metaklasy?
- Metaklasa definiuje reguły, według których klasy są tworzone.
- Można ją modyfikować, aby dodawać metody, atrybuty, walidacje lub zmieniać proces inicjalizacji klasy.



### 3. Tworzenie metaklasy
- Tworzy się je, dziedzicząc z `type`.
- Dwie kluczowe metody metaklas:
  - `__new__`: Tworzy nową klasę i umożliwia jej modyfikację.
  - `__init__`: Inicjalizuje klasę po jej utworzeniu.


In [ ]:
class MyMeta(type):
    def __new__(cls, name, bases, dct):
        print(f"Creating class {name} with metaclass {cls}")
        return super().__new__(cls, name, bases, dct)

class MyClass(metaclass=MyMeta):
    pass


Creating class MyClass with metaclass <class '__main__.MyMeta'>


### name, bases, class_dict

In [1]:
class AttributesMeta(type):
    def __new__(cls, name, bases, class_dict):
        print(name)         # str: name of class
        print(bases)        # tuple: with base classes that class inherits from.
        print(class_dict)   # dict: attributes and methods
        return super().__new__(cls, name, bases, class_dict)


class JitClass():
    ATTRIBUTE_ONE = "Value"


class ChildClass(JitClass, metaclass=AttributesMeta):
    invalid_attribute = "This will raise an error"

ChildClass
(<class '__main__.JitClass'>,)
{'__module__': '__main__', '__qualname__': 'ChildClass', 'invalid_attribute': 'This will raise an error'}


### 4. `__new__` i `__init__` w metaklasach
- `__new__`: Jest wywoływana przed utworzeniem klasy. Tu można modyfikować klasy lub dodać do nich dodatkowe atrybuty/metody.
- `__init__`: Wywoływana po utworzeniu klasy. Służy do dalszej inicjalizacji klasy.

In [ ]:
class ValidateAttributesMeta(type):
    def __new__(cls, name, bases, dct):
        if 'required_method' not in dct:
            raise TypeError(f"Class {name} must define 'required_method'")
        return super().__new__(cls, name, bases, dct)

class MyValidClass(metaclass=ValidateAttributesMeta):
    def required_method(self):
        pass

class MyInvalidClass(metaclass=ValidateAttributesMeta):
    # required_method = 3
    pass  # TypeError: Class MyInvalidClass must define 'required_method'



### 5. Przypisywanie metaklasy
- Aby przypisać metaklasę, należy użyć argumentu `metaclass` w definicji klasy.
- Jeśli metaklasa nie zostanie jawnie zdefiniowana, Python domyślnie użyje `type`.

In [ ]:
class MyClass(metaclass=MyMeta):
    pass


### 6. Zastosowania metaklas
1. **Walidacja klas**: Wymuszanie obecności określonych metod lub atrybutów.
2. **Automatyczne dodawanie metod lub atrybutów**: Ułatwia tworzenie wielu podobnych klas.
3. **Rejestracja klas**: Automatyczne dodawanie klas do centralnego rejestru.
4. **Modyfikacja klas**: Dynamiczne przekształcanie metod lub dodawanie dekoratorów.
5. **Wzorce projektowe**: Implementacja wzorców takich jak Singleton.


### 6. Zalety i wady metaklas
#### Zalety:
- Elastyczność w kontrolowaniu procesu tworzenia klas.
- Centralizacja logiki związanej z tworzeniem klas.
- Automatyzacja powtarzalnych wzorców w kodzie.
#### Wady:
- Wprowadzenie dodatkowej złożoności.
- Trudność w debugowaniu.
- Rzadkie zastosowanie – w wielu projektach są zbędne.

### 7. Czy warto używać metaklas?
Metaklasy są potężnym narzędziem, ale ich użycie należy ograniczyć do sytuacji, gdzie jest to naprawdę konieczne. W większości przypadków prostsze techniki, takie jak dekoratory czy dziedziczenie, są bardziej odpowiednie.

## Przykłady
Metaklasa sprawdzająca, czy każda klasa, która jej używa, posiada metodę `awesome_method`.

In [5]:
class MethodValidationMeta(type):
    def __new__(cls, name, bases, dct):
        if 'awesome_method' not in dct:
            raise TypeError(f"Class {name} must define 'awesome_method'")
        return super().__new__(cls, name, bases, dct)

class ValidClass(metaclass=MethodValidationMeta):
    def awesome_method(self):
        pass

# class InvalidClass(metaclass=MethodValidationMeta):
#     pass  # TypeError: Class InvalidClass must define 'required_method'


Metaklasa sprawdzająca, czy każda klasa, która jej używa, definiuje atrybut `awesome_attribute`.

In [6]:
class AttributeValidationMeta(type):
    def __new__(cls, name, bases, dct):
        if 'awesome_attribute' not in dct:
            raise TypeError(f"Class {name} must define 'awesome_attribute'")
        return super().__new__(cls, name, bases, dct)

class ValidClass(metaclass=AttributeValidationMeta):
    awesome_attribute = "I exist"

# class InvalidClass(metaclass=AttributeValidationMeta):
#     pass  # TypeError: Class InvalidClass must define 'required_attribute'


**Automatyczne dodawanie metod lub atrybutów**

Metaklasa automatycznie dodająca:

- Atrybut `default_value` ustawiony na `42`, jeśli nie jest zdefiniowany w klasie.
- Metodę `print_info`, która wypisuje nazwę klasy i wartość `default_value`.

In [ ]:
class AutoAddMeta(type):
    def __new__(cls, name, bases, dct):
        if 'default_value' not in dct:
            dct['default_value'] = 42

        if 'print_info' not in dct:
            def print_info(self):
                print(f"Class: {self.__class__.__name__}, default_value: {self.default_value}")
            dct['print_info'] = print_info

        return super().__new__(cls, name, bases, dct)

class MyClass(metaclass=AutoAddMeta):
    pass

class CustomClass(metaclass=AutoAddMeta):
    default_value = 99

obj1 = MyClass()
obj2 = CustomClass()

obj1.print_info()
obj2.print_info()


Class: MyClass, default_value: 42
Class: CustomClass, default_value: 99


Automatyczne dodawanie klas do centralnego rejestru.

In [ ]:
registry = {}

class RegisterMeta(type):
    def __new__(cls, name, bases, dct):
        cls_instance = super().__new__(cls, name, bases, dct)
        registry[name] = cls_instance
        return cls_instance

class BaseClass(metaclass=RegisterMeta):
    pass

class SubClass(BaseClass):
    pass

print(registry)


{'BaseClass': <class '__main__.BaseClass'>, 'SubClass': <class '__main__.SubClass'>}


Tworzymy metaklasę, która:

Automatycznie dodaje dekorator do wszystkich metod klas zdefiniowanych w klasie.
Dekorator loguje informacje o wywołaniu każdej metody (czyli nazwę metody i argumenty).

In [ ]:
import functools

def log_method_call(func):
    @functools.wraps(func)
    def wrapper(self, *args, **kwargs):
        print(f"Calling method: {func.__name__} with args: {args}, {kwargs}")
        return func(self, *args, **kwargs)
    return wrapper

class MethodModificationMeta(type):
    """Dodaje dekorator do każdej motody z klasy"""
    def __new__(cls, name, bases, dct):
        for attr_name, attr_value in dct.items():
            if callable(attr_value):
                dct[attr_name] = log_method_call(attr_value)
        return super().__new__(cls, name, bases, dct)

class MyClass(metaclass=MethodModificationMeta):
    def method1(self, x, y):
        return x + y

    def method2(self, a):
        return a * 2

obj = MyClass()

obj.method1(3, 4)
obj.method2(5)


Calling method: method1 with args: (3, 4), {}
Calling method: method2 with args: (5,), {}


10

Wzorce projektowe: Implementacja wzorców takich jak Singleton.

Singleton wymaga, aby niezależnie od liczby wywołań klasy, zawsze była zwracana ta sama instancja.

In [ ]:
class SingletonMeta(type):
    _instances = {}

    def __call__(cls, *args, **kwargs):
        if cls not in cls._instances:
            cls._instances[cls] = super().__call__(*args, **kwargs)
        return cls._instances[cls]

class SingletonClass(metaclass=SingletonMeta):
    pass

a = SingletonClass()
b = SingletonClass()
print(a is b)
print(id(a))
print(id(b))


True
4476047936
4476047936


Podczas: `obj = MyClass()`
Python automatycznie wywołuje jej metaklasę `__call__`.

Domyślna implementacja `__call__` (w `type`) wykonuje następujące kroki:
- Wywołuje `__new__`, aby utworzyć nowy obiekt.
- Następnie wywołuje `__init__`, aby zainicjalizować ten obiekt.
Możeszmy nadpisać `__call__` w metaklasie, aby zmienić zachowanie tego procesu.


### Ćwiczenia

#### Write a metaclass `StrictAttributesMeta` that ensures all class attributes are in uppercase. If any attribute is not uppercase, raise a `TypeError`.

In [ ]:
class StrictAttributesMeta(type):
    def __new__(cls, name, bases, class_dict):
        print(class_dict)   # for exploring solution.
        pass

class JitClass(metaclass=StrictAttributesMeta):
    ATTRIBUTE_ONE = "Value"
    ATTRIBUTE_TWO = "Another Value"


class InvalidClass(metaclass=StrictAttributesMeta):
    invalid_attribute = "This will raise an error"

#### Create a metaclass `LoggingMeta` that logs the name of every class it creates to the console.

In [ ]:
class LoggingMeta(type):
    pass

class MyClass(metaclass=LoggingMeta):
    pass

class AnotherClass(metaclass=LoggingMeta):
    pass

In [ ]:
class AttributesMeta(type):
    def __new__(cls, name, bases, class_dict):
        print(name)         # str: name of class
        print(bases)        # tuple: with base classes that class inherits from.
        print(class_dict)   # dict: attributes and methods
        return super().__new__(cls, name, bases, class_dict)


class JitClass():
    ATTRIBUTE_ONE = "Value"


class ChildClass(JitClass, metaclass=AttributesMeta):
    invalid_attribute = "This will raise an error"

ChildClass
(<class '__main__.JitClass'>,)
{'__module__': '__main__', '__qualname__': 'ChildClass', 'invalid_attribute': 'This will raise an error'}


#### Checmy mieć klasę która podczas inicjacji utworzy 2 albo 3 atrybuty, ale niee dokonać inicjlalizacij na 2 i 3 ustawić na None